#SILVER LAYER SCRIPT

### DATA ACCESS USING APP

### CREATE SECRET SCOPE

In [0]:
dbutils.secrets.get(scope="Datalake-Secrets" , key = "Application-ID")

In [0]:
spark.conf.set("fs.azure.account.auth.type.datalakeazureproject.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.datalakeazureproject.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.datalakeazureproject.dfs.core.windows.net", dbutils.secrets.get(scope="Datalake-Secrets" , key = "Application-ID"))
spark.conf.set("fs.azure.account.oauth2.client.secret.datalakeazureproject.dfs.core.windows.net", dbutils.secrets.get(scope="Datalake-Secrets" , key = "Client-Secret-value"))
spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalakeazureproject.dfs.core.windows.net", f"https://login.microsoftonline.com/{dbutils.secrets.get(scope='Datalake-Secrets' , key = 'Tenant-ID')}/oauth2/token")

### DATA LOADING

### Reading data

In [0]:
df_cus = spark.read.format('csv')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Customers')

In [0]:
df_prodcat = spark.read.format('csv')\
    .option('header',True)\
        .option('inferschema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Product_Categories')

In [0]:
df_prod = spark.read.format('csv')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Products')

In [0]:
df_returns = spark.read.format('csv')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Returns')

In [0]:
df_sales2015 = spark.read.format('csv')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Sales_2015')

In [0]:
df_sales2016 = spark.read.format('csv')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Sales_2016')

In [0]:
df_sales2017 = spark.read.format('csv')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Sales_2017')

In [0]:
df_terr = spark.read.format('csv')\
    .option('header',True)\
        .option ('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Territories')

In [0]:
df_prodsub = spark.read.format('csv')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/Product_Subcategories')

In [0]:
df_cal = spark.read.format('csv')\
    .option('header',True)\
        .option('inferSchema',True)\
            .load('abfss://bronze@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Calendar')

### TRANSFORMATIONS

### Calendar data

In [0]:
df_cal.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_cal = df_cal.withColumn('Month', month(col('Date')))\
    .withColumn('Year', year(col('Date')))
df_cal.display()

### Push Transformed Calendar data into Silver layer

In [0]:
df_cal.write.format('parquet')\
    .mode('append')\
        .option('path','abfss://silver@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Calendar')\
            .save()

### Customers data

In [0]:
df_cus.display()

In [0]:
df_cus.withColumn('FullName', concat(col('Prefix'),lit(' '),col('FirstName'),lit(' '),col('LastName'))).display()

### Effective concat using concat_ws( )

In [0]:
df_cus = df_cus.withColumn('FullName', concat_ws(' ',col('Prefix'),col('FirstName'),col('LastName')))

In [0]:
df_cus.display()

### Push Transformed Customer & prodcat data into Silver layer

In [0]:
df_cus.write.format('parquet')\
    .mode('append')\
        .option('path', 'abfss://silver@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Customers')\
            .save()

In [0]:
df_prodcat.display()

In [0]:
df_prodcat.write.format('parquet')\
    .mode('append')\
        .option('path', 'abfss://silver@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Product_Categories')\
            .save()

In [0]:
df_prodsub.display()

In [0]:
df_prodsub.write.format('parquet')\
    .mode('append')\
        .option('path', 'abfss://silver@datalakeazureproject.dfs.core.windows.net/Product_Subcategories')\
            .save()

### Products data

In [0]:
df_prod.display()

### Transformation

In [0]:
df_prod.display()

### Split ( ) to fetch index [0] on column

In [0]:
df_prod.withColumn('ProductSKU', split(col('ProductSKU'),'-')[0])\
    .withColumn('ProductName', split(col('ProductName'), ' ')[0])

In [0]:
df_prod = df_prod.withColumn('ProductSKU', split(col('ProductSKU'),'-')[0])\
    .withColumn('ProductName', split(col('ProductName'), ' ')[0])

In [0]:
df_prod.display()

### Push Transformed Products data into Silver layer :

In [0]:
df_prod.write.format('parquet')\
    .mode('append')\
        .option('path', 'abfss://silver@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Products')\
            .save()

### RETURNS & Terr data goes to silver layer

In [0]:
df_returns.display()

In [0]:
df_returns.write.format('parquet')\
    .mode('append')\
        .option('path', 'abfss://silver@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Returns')\
            .save()

In [0]:
df_terr.display()

In [0]:
df_terr.write.format('parquet')\
    .mode('append')\
        .option('path', 'abfss://silver@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Territories')\
            .save()

### Sales data

In [0]:
df_sales2015.display()

### Transformations

### to_timestamp()

In [0]:
df_sales2015 = df_sales2015.withColumn('StockDate', to_timestamp('StockDate'))

In [0]:
df_sales2015.display()

### Replace

In [0]:
df_sales2015 = df_sales2015.withColumn('OrderNumber', regexp_replace('OrderNumber','S','T'))

In [0]:
df_sales2015.display()

### Multiply

In [0]:
df_sales2015 = df_sales2015.withColumn('Multiplied', col('OrderLineItem')*col('OrderQuantity'))

## SALES ANALYSIS

### GroupBy()

In [0]:
df_sales2015 = df_sales2015.groupBy('OrderDate').agg(count('OrderNumber').alias('Total_order'))

Databricks visualization. Run in Databricks to view.

### sales data goes to silver layer

In [0]:
df_sales2015.write.format('parquet')\
    .mode('append')\
        .option('path','abfss://silver@datalakeazureproject.dfs.core.windows.net/AdventureWorks_Sales_2015')\
            .save()